# Giai đoạn 3 — chạy lại trên H100: loại bỏ yếu tố gây nhiễu

Notebook này **không phải** bản chạy lại y nguyên giai đoạn 2. Nó sửa đúng ba hạn chế đã
được nêu thẳng ở Chương 5 của báo cáo, theo thứ tự ưu tiên đã đề xuất ở Mục 5.4.

### Ba thứ được sửa

| # | Hạn chế ở v2 | Cách sửa ở v3 | Vì sao H100 mới làm được |
|---|---|---|---|
| 1 | Nhánh chưng cất chạy **batch 8**, nhánh đối chứng **batch 24** | Cả hai đều **batch 24** | T4 16 GB không đủ chỗ cho cả thầy lẫn trò ở batch 24; H100 80 GB thì thừa |
| 2 | Hệ số chưng cất `dis = 6,0` **chưa được quét thử** | Quét `dis ∈ {0,5 · 1 · 2 · 4 · 6}` | Mỗi giá trị là một lần train 50 chu kỳ — chỉ khả thi khi GPU đủ nhanh |
| 3 | Phép so INT8 ↔ FP32 **chưa tái lập độc lập** (FP32 không có sẵn khi đo lại) | Xuất **và** đo cả hai bản trong **cùng một phiên** | Không cần GPU, nhưng gộp vào đây để mọi số nằm trên một mặt bằng |

### Điều KHÔNG đổi — có chủ đích

Bộ đánh giá (`pycocotools`, cùng tệp chú thích COCO), hạt ngẫu nhiên 42, kích thước ảnh 640,
số chu kỳ 50, bộ tăng cường dữ liệu, và **batch của mô hình thầy vẫn là 16**. Giữ nguyên thầy
là cố ý: nhờ vậy mô hình thầy tái lập đúng bản cũ (mAP@0,5 = 0,9520), nên **biến số duy nhất
thay đổi giữa v2 và v3 chính là batch của nhánh chưng cất** — đúng thứ cần cô lập.

> ### Một đính chính kỹ thuật cho báo cáo
>
> Báo cáo v2 quy yếu tố gây nhiễu cho "kích thước lô 8 so với 24". Đọc mã nguồn Ultralytics thì
> câu chuyện tinh vi hơn một chút:
>
> ```python
> self.accumulate = max(round(self.args.nbs / self.batch_size), 1)
> ```
>
> Với `nbs = 64` mặc định: nhánh đối chứng batch 24 → `accumulate = 3` → lô **hiệu dụng 72**;
> nhánh chưng cất batch 8 → `accumulate = 8` → lô **hiệu dụng 64**. Tức về mặt *tối ưu hoá*, hai
> nhánh đã khá gần nhau (72 so với 64) chứ không lệch ba lần như con số 24/8 gợi ý.
>
> Nhưng **tích luỹ gradient không cào bằng được thống kê BatchNorm**: chuẩn hoá theo lô dùng
> trung bình và phương sai của *đúng lô đi qua mạng*, nên batch 8 và batch 24 vẫn cho hai chế
> độ chuẩn hoá khác nhau. Đây mới là phần nhiễu thật sự chưa kiểm soát được, và cách duy nhất
> để loại bỏ là cho hai nhánh chạy **cùng batch vật lý** — chính là điều v3 làm.
>
> Chi tiết này nên được đưa vào báo cáo dù kết quả v3 ra sao, vì nó làm phần "hạn chế" chính xác hơn.

### Thời gian ước tính trên H100

Thầy ~6 phút · đối chứng ~5 phút · chưng cất ~8 phút · quét 5 giá trị ~40 phút · xuất và đánh giá ~15 phút
→ **tổng khoảng 75 phút**. (Trên T4 cũ, riêng ba giai đoạn đầu đã mất ~3 giờ.)
Gấp về thời gian? Đặt `RUN_SWEEP = False` ở ô cấu hình — bỏ phần quét, còn **~35 phút**,
vẫn sửa trọn Ưu tiên 1 (tái lập ONNX) và Ưu tiên 2 (batch bằng nhau).

---
**Cách chạy:** Runtime → Change runtime type → **A100 hoặc H100** → chạy tuần tự từ trên xuống.
Ô cấu hình ngay dưới có `CONFIRM_FULL_TRAIN = False`; chạy thử `SMOKE_TEST` trọn vẹn trước rồi
mới bật lên `True`.


---
## §0 · Cấu hình và môi trường

In [ ]:
# ════════════════════════════ CẤU HÌNH ════════════════════════════
RUN_MODE = "SMOKE_TEST"        # "SMOKE_TEST" -> chay thu ~3 phut | "FULL_TRAIN" -> chay that
CONFIRM_FULL_TRAIN = False     # phai doi True khi RUN_MODE = "FULL_TRAIN"
RUN_STAGE = "ALL"              # ALL | TEACHER | BASELINE | KD | SWEEP | EXPORT_EVAL

SEED = 42
IMGSZ = 640

TEACHER_MODEL = "yolo26s.pt"
STUDENT_MODEL = "yolo26n.pt"

EPOCHS_TEACHER = 50
EPOCHS_STUDENT = 50
EPOCHS_SWEEP = 50              # phai bang EPOCHS_STUDENT thi quet moi so sanh duoc
PATIENCE = 12

# ─────────────────────────────────────────────────────────────────
#  THAY DOI COT LOI SO VOI v2
#  v2: BATCH_KD = 8  vs  BATCH_BASELINE = 24   -> yeu to gay nhieu
#  v3: ca hai deu 24                            -> loai bo hoan toan
#  Thay van giu batch 16 nhu v2 de tai lap dung mo hinh thay cu.
# ─────────────────────────────────────────────────────────────────
BATCH_TEACHER = 16             # GIU NGUYEN nhu v2 (co y)
BATCH_STUDENT = 24             # DUNG CHUNG cho ca doi chung lan chung cat
NBS = 64                       # nominal batch size, mac dinh cua Ultralytics

DIS_WEIGHT_MAIN = 6.0          # gia tri v2 da dung -> nhanh KD chinh
DIS_SWEEP = [0.5, 1.0, 2.0, 4.0]   # cac gia tri quet them (6.0 da co o nhanh chinh)
RUN_SWEEP = True               # False -> bo quet dis (~40 phut), van sua duoc Uu tien 1 & 2

# Optimizer. Mac dinh cua Ultralytics la "auto", va voi YOLO26 thi "auto" chon
# MuSGD — chinh la optimizer moi ma YOLO26 quang ba. Nhung v2 da ep AdamW, nen de
# so sanh duoc v2 <-> v3 thi PHAI giu AdamW. Muon thu MuSGD thi doi thanh "auto"
# hoac "MuSGD", nhung khi do ket qua KHONG con doi chieu truc tiep voi v2 duoc nua.
OPTIMIZER = "AdamW"            # "AdamW" (nhu v2) | "auto" | "MuSGD"

# Cache anh vao RAM: tren H100, GPU nhanh den muc NAP DU LIEU moi la nut co chai.
# Bo du lieu chi ~1,8 GB o dang giai nen nen cache duoc thoai mai. Khong anh huong
# ket qua — chi doi cho doc anh, tang cuong du lieu van chay lai moi chu ky.
CACHE_IMAGES = "ram"           # "ram" | "disk" | False

CONF_EVAL, CONF_VIZ, IOU_MATCH = 0.001, 0.25, 0.50
N_CALIB = 300
N_SPEED_WARMUP, N_SPEED_RUNS = 10, 100

# Nguong chap nhan — khai bao TRUOC khi chay, kiem tra tu dong o §6
THRESH = {
    "kd_not_worse_than_baseline": -0.005,
    "int8_max_drop": 0.015,
    "int8_min_size_reduction": 0.50,
}

# Ket qua v2 (Tesla T4, batch KD = 8) — dung de doi chieu o §6
V2_RESULTS = {
    "teacher":          {"map50": 0.9520, "map50_95": 0.7822, "ap_small": 0.6164, "ap_large": 0.8591},
    "student_baseline": {"map50": 0.9348, "map50_95": 0.7773, "ap_small": 0.5477, "ap_large": 0.8625},
    "student_kd":       {"map50": 0.9014, "map50_95": 0.7450, "ap_small": 0.4857, "ap_large": 0.8518},
    "onnx_fp32":        {"map50": 0.9079, "map50_95": 0.7473, "ap_small": 0.4620, "ap_large": 0.8470},
    "onnx_int8":        {"map50": 0.9041, "map50_95": 0.7437, "ap_small": 0.4370, "ap_large": 0.8390},
}
# ═══════════════════════════════════════════════════════════════════

import os, sys, json, time, random, shutil, subprocess, hashlib, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

if RUN_MODE == "FULL_TRAIN" and not CONFIRM_FULL_TRAIN:
    raise RuntimeError(
        "\n" + "=" * 68 + "\n"
        "  Ban dang yeu cau FULL_TRAIN (~75 phut tren H100).\n" + "=" * 68 + "\n"
        "  Hay chay SMOKE_TEST cho tron ven truoc, sau do dat:\n"
        "      CONFIRM_FULL_TRAIN = True\n" + "=" * 68)

SMOKE = RUN_MODE == "SMOKE_TEST"

if Path("/content").exists():
    PLATFORM, WORK = "colab", Path("/content/work")
    # Drive de CUOI: glob de quy toan bo MyDrive co the mat vai phut neu Drive lon.
    # kagglehub cache va /content duoc quet truoc nen truong hop thuong gap ket thuc nhanh.
    INPUT_ROOTS = [Path("/content"), Path("/root/.cache/kagglehub"), Path("/content/drive/MyDrive")]
elif Path("/kaggle/working").exists():
    PLATFORM, WORK, INPUT_ROOTS = "kaggle", Path("/kaggle/working"), [Path("/kaggle/input")]
else:
    PLATFORM, WORK, INPUT_ROOTS = "local", Path.cwd() / "work", [Path.cwd()]

OUT = WORK / "kd_v3" / ("smoke" if SMOKE else "full")
DATA = WORK / "data"
COCO = OUT / "01_dataset"
RUNS = OUT / "runs"
EXPORTS = OUT / "05_exports"
RESULTS = OUT / "06_results"
for d in (OUT, COCO, RUNS, EXPORTS, RESULTS,
          RESULTS / "metrics", RESULTS / "plots", RESULTS / "tables"):
    d.mkdir(parents=True, exist_ok=True)

_STAGES = ("TEACHER", "BASELINE", "KD", "SWEEP")
STAGE_DIR = {s: OUT / f"stage_{s.lower()}" for s in _STAGES}
for d in STAGE_DIR.values():
    d.mkdir(parents=True, exist_ok=True)

SMOKE_N = {"train": 48, "valid": 16, "test": 16}
if SMOKE:
    EPOCHS_TEACHER = EPOCHS_STUDENT = EPOCHS_SWEEP = 1
    BATCH_TEACHER = BATCH_STUDENT = 2
    IMGSZ, N_CALIB, N_SPEED_WARMUP, N_SPEED_RUNS = 160, 32, 2, 5
    DIS_SWEEP = [1.0]
    print("*" * 68)
    print("*  CHE DO CHAY THU — so lieu KHONG dung de bao cao")
    print("*  Muc tieu: xac nhan train / quet / export / danh gia deu chay tron ven")
    print("*" * 68)
else:
    print("=" * 68)
    print("  CHE DO HUAN LUYEN THAT — uoc tinh ~75 phut tren H100")
    print(f"  thay {EPOCHS_TEACHER} chu ky (batch {BATCH_TEACHER})")
    print(f"  tro  {EPOCHS_STUDENT} chu ky x 2 nhanh (batch {BATCH_STUDENT} — BANG NHAU)")
    if RUN_SWEEP:
        print(f"  quet {len(DIS_SWEEP)} gia tri dis: {DIS_SWEEP}   -> tong ~75 phut tren H100")
    else:
        print("  KHONG quet dis (RUN_SWEEP=False)          -> tong ~35 phut tren H100")
    print("=" * 68)

print(f"\nnen tang    : {PLATFORM}")
print(f"giai doan   : {RUN_STAGE}")
print(f"thu muc ra  : {OUT}")


def stage_enabled(s):
    return RUN_STAGE in ("ALL", s)


def stage_done(s):
    return (STAGE_DIR[s] / "stage_completed.json").exists()


def mark_done(s, info):
    (STAGE_DIR[s] / "stage_completed.json").write_text(json.dumps(info, indent=2))

### 0.1 Kiểm tra GPU

H100 là kiến trúc Hopper (`sm_90`). Ô này báo rõ đang được cấp GPU nào và bao nhiêu VRAM, vì
**toàn bộ lý do tồn tại của v3 là có đủ VRAM cho batch 24 ở nhánh chưng cất**. Nếu Colab chỉ
cấp T4 thì ô này sẽ cảnh báo và tự hạ về phương án tích luỹ gradient.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "=" * 68 + "\n  CHUA BAT GPU\n" + "=" * 68 + "\n"
        "  Colab : Runtime -> Change runtime type -> A100 hoac H100\n" + "=" * 68)

_cap = torch.cuda.get_device_capability(0)
_sm = f"sm_{_cap[0]}{_cap[1]}"
_name = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
DEVICE = 0

print(f"GPU        : {_name}  ({_sm})")
print(f"VRAM       : {VRAM_GB:.1f} GB")
print(f"torch      : {torch.__version__}")
print(f"kien truc  : {[a for a in torch.cuda.get_arch_list() if a.startswith('sm_')]}")

if _sm not in torch.cuda.get_arch_list():
    raise RuntimeError(f"Ban torch nay khong ho tro {_sm}. Cai lai torch moi hon.")

N_CPU = os.cpu_count() or 2
WORKERS = min(8, N_CPU)

# ─── Chon phuong an dat batch cho nhanh chung cat ───
# Nhanh KD phai nap CA thay lan tro vao VRAM. Uoc luong ~20-28 GB o batch 24,
# imgsz 640 (T4 16 GB tung OOM o batch > 8 nen v2 moi phai dung 8).
# NGUONG 30 chu KHONG phai 40: A100 40 GB bao total_memory ~39,5 GB — dat nguong
# 40 se khien A100 bi day nham xuong batch 8, am tham lap lai dung loi cua v2.
KD_BATCH = BATCH_STUDENT
KD_EXTRA_ARGS = {}
if not SMOKE and VRAM_GB < 30:
    # Khong du VRAM -> lui ve tich luy gradient de giu LO HIEU DUNG bang nhau.
    # Luu y: cach nay KHONG cao bang duoc thong ke BatchNorm — han che nay phai neu ro.
    KD_BATCH = 8
    KD_EXTRA_ARGS = {"nbs": NBS}
    print("\n" + "!" * 68)
    print(f"!  VRAM chi {VRAM_GB:.0f} GB — khong du cho batch {BATCH_STUDENT} o nhanh chung cat.")
    print(f"!  Lui ve batch {KD_BATCH} + tich luy gradient (nbs={NBS}).")
    print("!  CANH BAO: cach nay can bang duoc lo HIEU DUNG nhung KHONG can bang")
    print("!  duoc thong ke BatchNorm -> yeu to gay nhieu VAN CON. Muon loai han")
    print("!  phai chay tren GPU >= 30 GB (A100 40GB / H100).")
    print("!" * 68)
else:
    print(f"\nVRAM du -> nhanh chung cat chay batch {KD_BATCH}, BANG nhanh doi chung.")
    print("Day chinh la thay doi cot loi cua v3 so voi v2.")

print(f"\nworkers    : {WORKERS}")
print(f"accumulate : doi chung = {max(round(NBS / BATCH_STUDENT), 1)}"
      f" | chung cat = {max(round(NBS / KD_BATCH), 1)}")

### 0.2 Cài thư viện và kiểm tra chưng cất có được hỗ trợ không

Bước chặn bắt buộc. Nếu phiên bản Ultralytics không có tham số `distill_model`, notebook phải
dừng ngay — **tuyệt đối không âm thầm chuyển sang một cơ chế chưng cất tự viết**, vì như vậy
kết quả không còn so được với v2.

In [ ]:
import torchvision

CONSTRAINTS = OUT / "pip-constraints.txt"
CONSTRAINTS.write_text(
    f"torch=={torch.__version__.split('+')[0]}\n"
    f"torchvision=={torchvision.__version__.split('+')[0]}\n")


def pip_install(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-c", str(CONSTRAINTS)] + args.split(), check=False)


pip_install("-U ultralytics")
pip_install("onnx onnxruntime onnxslim")
pip_install("pycocotools")

import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT

print("ultralytics :", ultralytics.__version__)

_missing = [k for k in ("distill_model", "dis") if k not in DEFAULT_CFG_DICT]
if _missing:
    raise RuntimeError(
        "\n" + "=" * 68 + "\n  PHIEN BAN ULTRALYTICS KHONG HO TRO CHUNG CAT\n" + "=" * 68 + "\n"
        f"  Thieu tham so: {_missing}\n"
        "  Nang cap:  pip install -U ultralytics\n" + "=" * 68)

print("chung cat   : DUOC HO TRO (co distill_model + dis)")
print("nbs mac dinh:", DEFAULT_CFG_DICT.get("nbs"))

In [ ]:
import numpy as np


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()
(OUT / "environment.json").write_text(json.dumps({
    "platform": PLATFORM, "torch": torch.__version__,
    "torchvision": torchvision.__version__, "ultralytics": ultralytics.__version__,
    "gpu": _name, "sm": _sm, "vram_gb": round(VRAM_GB, 1), "n_cpu": N_CPU,
    "run_mode": RUN_MODE, "seed": SEED,
    "batch_teacher": BATCH_TEACHER, "batch_student": BATCH_STUDENT,
    "batch_kd_actual": KD_BATCH, "kd_extra_args": KD_EXTRA_ARGS,
}, indent=2))
print("seed =", SEED, "| da ghi environment.json")

---
## §1 · Dữ liệu

Giữ nguyên bộ `pkdarabi/cardetection`, 15 lớp gốc, **không gộp và không ánh xạ lại** — điều
kiện bắt buộc để kết quả v3 ghép chung bảng được với v2.

### 1.1 Lấy dataset

Ba đường, thử lần lượt: (a) đã có sẵn trong phiên, (b) `kagglehub` tải trực tiếp, (c) Google
Drive. Sau khi tìm thấy, dataset được **sao chép sang thư mục ghi được** — bắt buộc, vì
Ultralytics cần ghi tệp `labels.cache` cạnh thư mục nhãn.

In [ ]:
import yaml
from PIL import Image

KAGGLE_SLUG = "pkdarabi/cardetection"


def score_dir(d):
    """Cham diem mot thu muc ung vien chua data.yaml."""
    have = sum((d / s / "images").is_dir() for s in ("train", "valid", "val", "test"))
    n = len(list((d / "train" / "images").glob("*"))) if (d / "train" / "images").is_dir() else 0
    return have, n


def find_dataset_root(roots):
    cands = []
    for root in roots:
        if not root.exists():
            continue
        for y in root.glob("**/data.yaml"):
            have, n = score_dir(y.parent)
            if have >= 2 and n > 0:
                cands.append((have, n, y.parent))
    return max(cands)[2] if cands else None


SRC = find_dataset_root(INPUT_ROOTS)

if SRC is None:
    print("Khong thay dataset trong phien -> thu tai bang kagglehub...")
    try:
        pip_install("-U kagglehub")
        import kagglehub
        p = Path(kagglehub.dataset_download(KAGGLE_SLUG))
        print("kagglehub tai ve:", p)
        SRC = find_dataset_root([p])
    except Exception as ex:
        print("kagglehub that bai:", ex)

if SRC is None and PLATFORM == "colab":
    print("\nThu gan Google Drive...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SRC = find_dataset_root([Path("/content/drive/MyDrive")])
    except Exception as ex:
        print("gan Drive that bai:", ex)

if SRC is None:
    raise FileNotFoundError(
        "\n" + "=" * 68 + "\n  KHONG TIM THAY DATASET\n" + "=" * 68 + "\n"
        f"  Cach 1: dang nhap Kaggle roi chay lai (kagglehub tai '{KAGGLE_SLUG}')\n"
        "  Cach 2: tai thu cong roi giai nen vao /content/\n"
        "  Cach 3: dat thu muc dataset vao Google Drive roi chay lai\n"
        "  Thu muc phai co: data.yaml + train/images + valid/images + test/images\n"
        + "=" * 68)

print("nguon dataset :", SRC)

VALID_DIR = "valid" if (SRC / "valid").is_dir() else "val"

if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Chep theo TUNG CAP anh-nhan. Cat rieng hai thu muc se sinh ra nhan mo coi
# (nhan khong co anh di kem) khi chay che do thu — Ultralytics se canh bao
# va so luong dem duoc se lech.
for s in ("train", VALID_DIR, "test"):
    src_i, src_l = SRC / s / "images", SRC / s / "labels"
    dst_i, dst_l = DATA / s / "images", DATA / s / "labels"
    dst_i.mkdir(parents=True, exist_ok=True)
    dst_l.mkdir(parents=True, exist_ok=True)
    if not src_i.is_dir():
        continue
    imgs = sorted(f for f in src_i.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXT)
    if SMOKE:
        key = "valid" if s == VALID_DIR else s
        imgs = imgs[: SMOKE_N.get(key, 16)]
    for f in imgs:
        shutil.copy2(f, dst_i / f.name)
        lp = src_l / (f.stem + ".txt")
        if lp.exists():
            shutil.copy2(lp, dst_l / lp.name)

_y = yaml.safe_load((SRC / "data.yaml").read_text())
_names = _y.get("names")
CLASSES = ([_names[i] for i in sorted(_names)] if isinstance(_names, dict) else list(_names))
NUM_CLASSES = len(CLASSES)

print(f"da sao chep sang: {DATA}")
for s in ("train", VALID_DIR, "test"):
    n_i = len(list((DATA / s / "images").glob("*")))
    n_l = len(list((DATA / s / "labels").glob("*.txt")))
    print(f"   {s:6s}: {n_i:5d} anh | {n_l:5d} nhan")
print(f"so lop: {NUM_CLASSES}")

### 1.2 Chuyển nhãn YOLO → COCO cho bộ đánh giá

Trường `area` không chỉ mang tính mô tả: **pycocotools dùng chính nó để phân nhóm nhỏ / vừa /
lớn** khi tính `AP` theo kích thước — chỉ số trung tâm của cả đề tài.

In [ ]:
def split_dir(s):
    return DATA / (VALID_DIR if s == "valid" else s)


def label_path(ip):
    return ip.parent.parent / "labels" / (ip.stem + ".txt")


def read_yolo_label(p):
    if not p.exists():
        return []
    out = []
    for line in p.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        c = int(float(parts[0]))
        xc, yc, w, h = (float(v) for v in parts[1:5])
        if w > 0 and h > 0:
            out.append((c, xc, yc, w, h))
    return out


def list_images(split):
    d = split_dir(split) / "images"
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted(p for p in d.iterdir() if p.suffix.lower() in exts) if d.is_dir() else []


def to_coco(split):
    imgs, anns, aid = [], [], 1
    for iid, ip in enumerate(list_images(split), 1):
        with Image.open(ip) as im:
            W, H = im.size
        imgs.append({"id": iid, "file_name": ip.name, "width": W, "height": H,
                     "abs_path": str(ip)})
        for c, xc, yc, w, h in read_yolo_label(label_path(ip)):
            aw, ah = w * W, h * H
            anns.append({"id": aid, "image_id": iid, "category_id": c + 1,
                         "bbox": [(xc - w / 2) * W, (yc - h / 2) * H, aw, ah],
                         "area": aw * ah, "iscrowd": 0})
            aid += 1
    return {"images": imgs, "annotations": anns,
            "categories": [{"id": i + 1, "name": n} for i, n in enumerate(CLASSES)]}


for s in ("train", "valid", "test"):
    d = to_coco(s)
    (COCO / f"instances_{s}.json").write_text(json.dumps(d))
    print(f"{s:6s}: {len(d['images']):5d} anh | {len(d['annotations']):5d} hop")

### 1.3 Kiểm toán rò rỉ dữ liệu giữa các tập

Bộ dữ liệu này **có rò rỉ**: v2 đo được 65/638 ảnh kiểm tra (10,2%) có bản sao trong tập huấn
luyện, và **71,3% số hộp trong phần rò rỉ thuộc nhóm vật thể nhỏ** — đúng nhóm mà báo cáo quan
tâm nhất. Notebook không dừng, mà tách sẵn tập con sạch rồi báo cáo **song song hai con số**.

In [ ]:
import re

RF_PAT = re.compile(r"^(.*?)\.rf\.[0-9a-f]+\.\w+$")


def source_stem(name):
    m = RF_PAT.match(name)
    return m.group(1) if m else Path(name).stem


def sha256_of(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while chunk := f.read(buf):
            h.update(chunk)
    return h.hexdigest()


by_stem, by_hash = {}, {}
for s in ("train", "valid", "test"):
    for ip in list_images(s):
        by_stem.setdefault(source_stem(ip.name), set()).add(s)
        by_hash.setdefault(sha256_of(ip), set()).add(s)

cross_stem = {k: v for k, v in by_stem.items() if len(v) > 1}
cross_hash = {k: v for k, v in by_hash.items() if len(v) > 1}

TEST_META_TMP = json.loads((COCO / "instances_test.json").read_text())
leak_ids = []
for im in TEST_META_TMP["images"]:
    st = source_stem(im["file_name"])
    if st in cross_stem and "train" in cross_stem[st]:
        leak_ids.append(im["id"])

LEAKED_TEST_IDS = sorted(leak_ids)
ALL_TEST_IDS = [im["id"] for im in TEST_META_TMP["images"]]
CLEAN_TEST_IDS = sorted(set(ALL_TEST_IDS) - set(LEAKED_TEST_IDS))

leak_report = {
    "cross_split_groups": len(cross_stem),
    "byte_identical_groups": len(cross_hash),
    "leaked_test_images": len(LEAKED_TEST_IDS),
    "test_images": len(ALL_TEST_IDS),
    "leak_ratio": round(len(LEAKED_TEST_IDS) / max(len(ALL_TEST_IDS), 1), 4),
    "clean_test_images": len(CLEAN_TEST_IDS),
}
(RESULTS / "metrics" / "data_leakage.json").write_text(json.dumps(leak_report, indent=2))

for k, v in leak_report.items():
    print(f"{k:24s}: {v}")
if not SMOKE:
    print("\n(v2 do duoc: 155 nhom / 101 trung byte / 65 anh ro ri / 573 anh sach)")

### 1.4 Hai tệp YAML: một để huấn luyện, một để hiệu chuẩn INT8

> **Bẫy tinh vi.** Khi lượng tử hoá INT8, Ultralytics đọc split `val` trong tệp YAML để lấy ảnh
> hiệu chuẩn. Đưa `data.yaml` thường vào thì nó hiệu chuẩn bằng **tập kiểm định** — tức là rò rỉ.
> Vì vậy phải tạo YAML riêng, trong đó `val` **trỏ tới ảnh tập huấn luyện**.

In [ ]:
DATA_YAML = OUT / "data_train.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA), "train": "train/images",
    "val": f"{VALID_DIR}/images", "test": "test/images",
    "nc": NUM_CLASSES, "names": {i: c for i, c in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True))

CALIB_DIR = OUT / "calib"
if CALIB_DIR.exists():
    shutil.rmtree(CALIB_DIR)
(CALIB_DIR / "images").mkdir(parents=True)
(CALIB_DIR / "labels").mkdir(parents=True)

# Lay anh hieu chuan tu TAP HUAN LUYEN, rai deu theo lop de khong lech phan bo.
_train_imgs = list_images("train")
_by_cls = {}
for ip in _train_imgs:
    for c, *_ in read_yolo_label(label_path(ip)):
        _by_cls.setdefault(c, []).append(ip)

_picked, _seen = [], set()
_rng = random.Random(SEED)
_per = max(1, N_CALIB // max(len(_by_cls), 1))
for c in sorted(_by_cls):
    pool = _by_cls[c][:]
    _rng.shuffle(pool)
    for ip in pool[:_per]:
        if ip not in _seen:
            _seen.add(ip)
            _picked.append(ip)

_rest = [p for p in _train_imgs if p not in _seen]
_rng.shuffle(_rest)
_picked += _rest[: max(0, N_CALIB - len(_picked))]
_picked = _picked[:N_CALIB]

for ip in _picked:
    shutil.copy2(ip, CALIB_DIR / "images" / ip.name)
    lp = label_path(ip)
    if lp.exists():
        shutil.copy2(lp, CALIB_DIR / "labels" / lp.name)

CALIB_YAML = OUT / "data_calib.yaml"
CALIB_YAML.write_text(yaml.safe_dump({
    "path": str(CALIB_DIR), "train": "images", "val": "images",
    "nc": NUM_CLASSES, "names": {i: c for i, c in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True))

print(f"data_train.yaml : {DATA_YAML}")
print(f"data_calib.yaml : {CALIB_YAML}  ({len(_picked)} anh tu TAP HUAN LUYEN)")

---
## §2 · Huấn luyện — lần này hai nhánh học trò có batch bằng nhau

Cấu hình chung cho cả ba, chỉ khác **đúng những gì phải khác**.

> **Vì sao tắt lật ảnh (`fliplr=0.0`, `flipud=0.0`):** biển "rẽ trái" lật ngang thành biển
> "rẽ phải" — phép tăng cường này dạy mô hình một điều sai sự thật. Giữ nhất quán với v2.

In [ ]:
from ultralytics import YOLO

COMMON = dict(
    data=str(DATA_YAML), imgsz=IMGSZ, seed=SEED, deterministic=True,
    patience=PATIENCE, optimizer=OPTIMIZER, lr0=0.001, lrf=0.01, cos_lr=True,
    amp=True, workers=WORKERS, device=DEVICE, val=True, plots=True, exist_ok=True,
    save_period=-1, project=str(RUNS), cache=CACHE_IMAGES,
    # Dat nbs TUONG MINH thay vi dua vao mac dinh cua thu vien: neu ban Ultralytics
    # sau doi mac dinh, accumulate se doi theo va moi con so "lo hieu dung" bao cao
    # o day thanh sai ma khong ai biet.
    nbs=NBS,
    # tang cuong vua phai, KHONG lat anh
    flipud=0.0, fliplr=0.0, degrees=3.0, translate=0.10, scale=0.30,
    shear=0.0, perspective=0.0, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    mosaic=1.0, mixup=0.0, close_mosaic=10 if not SMOKE else 0,
)


print(f"optimizer  : {OPTIMIZER}"
      + ("   (nhu v2 — so sanh truc tiep duoc)" if OPTIMIZER == "AdamW"
         else "   !! KHAC v2 — ket qua KHONG doi chieu truc tiep voi v2 duoc"))
print(f"cache anh  : {CACHE_IMAGES}")
print(f"fliplr     : {COMMON['fliplr']}  (TAT — bien 're trai' lat ngang thanh 're phai')")


def n_epochs_done(run_name):
    """So chu ky THUC SU da chay, doc tu results.csv (co the < epochs neu dung som)."""
    csv = RUNS / run_name / "results.csv"
    if not csv.exists():
        return 0
    try:
        return max(sum(1 for _ in csv.open()) - 1, 0)
    except Exception:
        return 0


def train_one(run_name, model_name, epochs, batch, extra=None, stage=None):
    """Huan luyen mot lan chay; chi bo qua khi lan truoc da chay XONG."""
    best = RUNS / run_name / "weights" / "best.pt"
    marker = RUNS / run_name / "v3_completed.json"

    # Ultralytics ghi de best.pt sau MOI chu ky, nen chi kiem tra best.exists()
    # la khong du: mot phien Colab bi ngat giua chung van de lai best.pt cua mot
    # mo hinh huan luyen do dang, va lan chay sau se am tham dung no nhu that.
    # Vi vay moi lan chay xong moi ghi dau rieng; thieu dau = phai chay lai.
    if best.exists() and marker.exists():
        print(f"[{run_name}] da hoan thanh o phien truoc -> bo qua"
              f"  ({n_epochs_done(run_name)} chu ky)")
        return best
    if best.exists() and not marker.exists():
        print(f"[{run_name}] CO best.pt nhung THIEU dau hoan thanh"
              f" ({n_epochs_done(run_name)} chu ky) -> phien truoc bi ngat, huan luyen LAI")

    if stage is not None and not stage_enabled(stage):
        if best.exists() and marker.exists():
            print(f"[{run_name}] ngoai RUN_STAGE nhung da hoan thanh -> dung lai")
            return best
        raise RuntimeError(
            f"[{run_name}] chua hoan thanh va ngoai RUN_STAGE={RUN_STAGE}")

    seed_everything()
    kw = dict(COMMON, epochs=epochs, batch=batch, name=run_name)
    if extra:
        kw.update(extra)
    acc = max(round(kw.get("nbs", NBS) / batch), 1)
    print(f"\n[{run_name}] {model_name} | {epochs} chu ky | batch {batch}"
          f" | accumulate {acc} -> lo hieu dung {batch * acc}")
    t0 = time.time()
    m = YOLO(model_name)
    m.train(**kw)
    mins = (time.time() - t0) / 60
    done = n_epochs_done(run_name)
    info = {"run": run_name, "model": model_name, "epochs_requested": epochs,
            "epochs_actual": done, "early_stopped": done < epochs,
            "batch": batch, "accumulate": acc, "effective_batch": batch * acc,
            "minutes": round(mins, 1), "best": str(best)}
    marker.write_text(json.dumps(info, indent=2))
    if stage is not None:
        mark_done(stage, info)
    print(f"[{run_name}] xong trong {mins:.1f} phut | {done}/{epochs} chu ky"
          + ("  (DUNG SOM)" if done < epochs else "") + f" -> {best}")
    return best

### 2.1 Thầy — `yolo26s`, batch 16

Giữ nguyên batch 16 như v2 là **có chủ đích**: mô hình thầy phải tái lập đúng bản cũ để biến số
duy nhất thay đổi giữa hai lần chạy nằm ở nhánh chưng cất.

In [ ]:
TEACHER_BEST = train_one("teacher", TEACHER_MODEL, EPOCHS_TEACHER,
                         BATCH_TEACHER, stage="TEACHER")

### 2.2 Trò đối chứng — `yolo26n`, batch 24

Mốc so sánh. Mọi tham số giống hệt trò chưng cất, **trừ** việc không có `distill_model`.

In [ ]:
BASELINE_BEST = train_one("baseline", STUDENT_MODEL, EPOCHS_STUDENT,
                          BATCH_STUDENT, stage="BASELINE")

### 2.3 Trò chưng cất — `yolo26n`, **cũng batch 24**

Đây là ô quan trọng nhất của cả notebook. Ở v2, ô này chạy batch 8 vì T4 không đủ chỗ cho cả
thầy lẫn trò; chênh lệch quan sát được vì thế không quy hết về chưng cất được.

Bây giờ cả hai nhánh dùng **cùng batch vật lý, cùng accumulate, cùng thống kê BatchNorm** —
khác biệt duy nhất còn lại đúng bằng hai tham số `distill_model` và `dis`.

In [ ]:
KD_BEST = train_one(
    "kd", STUDENT_MODEL, EPOCHS_STUDENT, KD_BATCH,
    extra={"distill_model": str(TEACHER_BEST), "dis": DIS_WEIGHT_MAIN, **KD_EXTRA_ARGS},
    stage="KD")

print("\n" + "=" * 68)
print("  DOI CHIEU DIEU KIEN HAI NHANH HOC TRO")
print("=" * 68)
print(f"  doi chung : batch {BATCH_STUDENT} | accumulate {max(round(NBS/BATCH_STUDENT),1)}")
print(f"  chung cat : batch {KD_BATCH} | accumulate {max(round(NBS/KD_BATCH),1)}")
if KD_BATCH == BATCH_STUDENT:
    print("  -> batch BANG NHAU. Yeu to gay nhieu chinh cua v2 da duoc loai bo.")
else:
    print("  -> lo hieu dung bang nhau nhung BatchNorm VAN khac. Phai neu o phan han che.")

# Dung som (patience) co the khien hai nhanh chay khac so chu ky — dieu do se
# TAI LAP lai dung loai yeu to gay nhieu ma v3 sinh ra de loai bo. Phai kiem tra
# chu khong duoc mac dinh la "cung 50 chu ky".
_e_bl, _e_kd = n_epochs_done("baseline"), n_epochs_done("kd")
print(f"\n  so chu ky THUC SU chay:  doi chung {_e_bl}  |  chung cat {_e_kd}"
      f"   (yeu cau {EPOCHS_STUDENT}, patience {PATIENCE})")
if _e_bl != _e_kd:
    print("  !! CANH BAO: hai nhanh KHONG chay cung so chu ky (dung som khac nhau).")
    print("     Day la mot yeu to gay nhieu MOI, phai neu ro trong bao cao —")
    print("     hoac chay lai voi patience=0 de tat dung som hoan toan.")
else:
    print("  -> cung so chu ky. Khong phat sinh yeu to gay nhieu tu dung som.")
print("=" * 68)

### 2.4 Xác minh chưng cất **thật sự** đã được kích hoạt

Truyền tham số vào không có nghĩa là nó chạy. Bằng chứng duy nhất đáng tin là cột `dis_loss`
xuất hiện trong nhật ký huấn luyện — và **không tồn tại** ở nhánh đối chứng. Nếu thiếu, toàn bộ
thí nghiệm vô nghĩa, nên dừng hẳn thay vì báo cáo một kết quả sai.

In [ ]:
import pandas as pd

_kd_csv = RUNS / "kd" / "results.csv"
assert _kd_csv.exists(), f"Khong thay {_kd_csv}"
_kd_df = pd.read_csv(_kd_csv)
_kd_df.columns = [c.strip() for c in _kd_df.columns]
_dis_cols = [c for c in _kd_df.columns if "dis" in c.lower() and "loss" in c.lower()]

print("cac cot trong results.csv cua nhanh KD:")
print("   " + ", ".join(_kd_df.columns))

if not _dis_cols:
    raise RuntimeError(
        "\n" + "=" * 68 + "\n  CHUNG CAT KHONG DUOC KICH HOAT\n" + "=" * 68 + "\n"
        "  Khong thay cot 'dis_loss' -> tham so distill_model da bi bo qua am tham.\n"
        "=" * 68)

_c = _dis_cols[0]
_first, _last = float(_kd_df[_c].iloc[0]), float(_kd_df[_c].iloc[-1])
print(f"\nXAC NHAN: chung cat da chay. Cot '{_c}': {_first:.4f} -> {_last:.4f}")

# Doi chieu: nhanh doi chung KHONG duoc co cot nay
_bl_csv = RUNS / "baseline" / "results.csv"
if _bl_csv.exists():
    _bl_df = pd.read_csv(_bl_csv)
    _bl_df.columns = [c.strip() for c in _bl_df.columns]
    _bl_dis = [c for c in _bl_df.columns if "dis" in c.lower() and "loss" in c.lower()]
    print(f"nhanh doi chung co cot dis_loss? {'CO — BAT THUONG!' if _bl_dis else 'KHONG (dung nhu ky vong)'}")

(RESULTS / "metrics" / "distill_activation.json").write_text(json.dumps({
    "column": _c, "first_epoch": _first, "last_epoch": _last,
    "monotonic_decrease": bool(_last < _first),
    "baseline_has_dis_loss": bool(_bl_csv.exists() and _bl_dis),
}, indent=2))

---
## §3 · Quét hệ số chưng cất

Ưu tiên 3 trong Mục 5.4 của báo cáo. Giá trị `dis = 6,0` ở v2 là **mặc định của thư viện, chưa
từng được thử giá trị nào khác** — nên chưa loại trừ được giả thuyết "thành phần bắt chước lấn
át tín hiệu từ nhãn cứng".

Mỗi giá trị là một lần huấn luyện đầy đủ, giống hệt nhánh KD chính, **chỉ khác đúng `dis`**.
Nhánh `dis = 6,0` đã chạy ở §2.3 nên không lặp lại.

In [ ]:
sweep_runs = {DIS_WEIGHT_MAIN: KD_BEST}

if RUN_SWEEP and stage_enabled("SWEEP"):
    for i, dw in enumerate(DIS_SWEEP, 1):
        print(f"\n{'#' * 68}\n#  QUET {i}/{len(DIS_SWEEP)} — dis = {dw}\n{'#' * 68}")
        sweep_runs[dw] = train_one(
            f"kd_dis{str(dw).replace('.', 'p')}", STUDENT_MODEL, EPOCHS_SWEEP, KD_BATCH,
            extra={"distill_model": str(TEACHER_BEST), "dis": dw, **KD_EXTRA_ARGS})
    mark_done("SWEEP", {"values": sorted(sweep_runs), "runs": {str(k): str(v) for k, v in sweep_runs.items()}})
else:
    print(f"bo qua quet (RUN_SWEEP={RUN_SWEEP}, RUN_STAGE={RUN_STAGE})"
          " — chi con nhanh dis=6.0 tu §2.3")

print("\ncac nhanh chung cat da co:")
for dw in sorted(sweep_runs):
    print(f"   dis={dw:<5} -> {sweep_runs[dw]}")

---
## §4 · Xuất ONNX — lần này FP32 và INT8 đo trong **cùng một phiên**

Ưu tiên 1 trong Mục 5.4, và là hạn chế đáng ngại nhất của v2: bản FP32 không còn khi đo lại,
nên **phép so INT8 ↔ FP32 chỉ dựa trên một lần đo duy nhất**. Ở đây cả hai được xuất và đo
liền nhau trên cùng một máy, cùng một `onnxruntime`, nên kết luận về lượng tử hoá cuối cùng
cũng đạt độ tin cậy ngang phần chưng cất.

> **`end2end=False` là lựa chọn có chủ đích.** YOLO26 mặc định xuất ở chế độ NMS-free với NMS
> nhúng sẵn trong đồ thị. Chế độ đó nhanh hơn khi triển khai nhưng **không so trực tiếp được**
> với các cấu hình PyTorch dùng hậu xử lý NMS cổ điển. Giữ `end2end=False` để cả bảng nằm trên
> cùng một mặt bằng.

In [ ]:
def export_model(weights, fmt, quant=None, data=None, tag=""):
    """Xuat mo hinh; tu xu ly truong hop thu muc nguon chi doc."""
    weights = Path(weights)
    if not os.access(weights.parent, os.W_OK):
        local = EXPORTS / f"src_{weights.name}"
        if not local.exists():
            shutil.copy(weights, local)
        print(f"   (nguon chi doc -> dung ban sao {local.name})")
        weights = local

    m = YOLO(str(weights))
    kw = dict(format=fmt, imgsz=IMGSZ, batch=1, dynamic=False, device="cpu")
    # end2end=False giu bo cuc tensor co dien de so duoc voi v2. Nhung notebook nay
    # cai "-U ultralytics" (ban moi nhat), neu ban moi bo key nay thi truyen vao se
    # crash — nen chi truyen khi thu vien con khai bao no.
    if "end2end" in DEFAULT_CFG_DICT:
        kw["end2end"] = False
    else:
        print("   !! ban ultralytics nay khong con 'end2end' — kiem tra ky tensor dau ra")
    if fmt == "onnx":
        kw.update(opset=19, simplify=True)
    if data:
        kw["data"] = str(data)

    # API moi (quantize=) truoc, lui ve co cu (int8=/half=) neu can
    trials = {16: [{"quantize": 16}, {"half": True}],
              8: [{"quantize": 8}, {"int8": True}]}.get(quant, [{}])
    last = None
    for extra in trials:
        try:
            out = Path(m.export(**dict(kw, **extra)))
            dst = EXPORTS / f"kd{('_' + tag) if tag else ''}{out.suffix}"
            if out.resolve() != dst.resolve():
                shutil.move(str(out), dst)
            print(f"   {dst.name:28s} {dst.stat().st_size / 1024**2:7.2f} MB   ({extra or 'mac dinh'})")
            return dst
        except Exception as ex:
            last = ex
            print(f"   thu {extra} that bai: {type(ex).__name__}")
    raise RuntimeError(f"Xuat that bai voi moi API: {last}")


print("Xuat ONNX FP32 (khong luong tu hoa)")
KD_ONNX_FP32 = export_model(KD_BEST, "onnx", tag="fp32")

print("\nXuat ONNX INT8 (hieu chuan bang anh TAP HUAN LUYEN)")
KD_ONNX_INT8 = export_model(KD_BEST, "onnx", quant=8, data=CALIB_YAML, tag="int8")

_r = 1 - KD_ONNX_INT8.stat().st_size / KD_ONNX_FP32.stat().st_size
print(f"\nINT8 nho hon FP32: {_r:.1%}")
print("CA HAI ban deu xuat trong cung phien nay -> phep so sanh tai lap duoc.")

### 4.1 Kiểm chứng tệp đã xuất

Xuất xong không có nghĩa là dùng được. Kiểm tra bốn thứ: đồ thị hợp lệ, nạp được, chạy được
một ảnh, và kết quả không chứa `NaN`.

In [ ]:
import onnx
import onnxruntime as ort

_chk = {}
for nm, pth in [("ONNX FP32", KD_ONNX_FP32), ("ONNX INT8", KD_ONNX_INT8)]:
    rec = {"path": str(pth), "size_mb": round(pth.stat().st_size / 1024**2, 2)}
    try:
        mo = onnx.load(str(pth))
        onnx.checker.check_model(mo)
        ops = [n.op_type for n in mo.graph.node]
        rec.update(graph_valid=True, n_nodes=len(ops),
                   n_quant_nodes=sum(o in ("QuantizeLinear", "DequantizeLinear",
                                           "QLinearConv", "QLinearMatMul") for o in ops))
        rec["mixed_precision"] = 0 < rec["n_quant_nodes"] < len(ops)
        sess = ort.InferenceSession(str(pth), providers=["CPUExecutionProvider"])
        inp = sess.get_inputs()[0]
        y = sess.run(None, {inp.name: np.zeros([1, 3, IMGSZ, IMGSZ], dtype=np.float32)})
        rec["output_shapes"] = [list(o.shape) for o in y]
        rec["has_nan"] = bool(any(np.isnan(o).any() for o in y))
        rec["runnable"] = True
    except Exception as ex:
        rec.update(graph_valid=False, runnable=False, error=f"{type(ex).__name__}: {ex}")
    _chk[nm] = rec
    print(f"\n{nm}")
    for k, v in rec.items():
        if k != "path":
            print(f"   {k:18s}: {v}")

(RESULTS / "metrics" / "onnx_check.json").write_text(json.dumps(_chk, indent=2))
print(f"\nonnxruntime: {ort.__version__} | providers: {ort.get_available_providers()}")

---
## §5 · Đánh giá bằng một bộ đo duy nhất

Mọi cấu hình đều nạp qua `YOLO(...)`, nên **đường tiền xử lý và hậu xử lý hoàn toàn giống nhau**;
khác biệt duy nhất là trọng số và định dạng. Đo bằng `pycocotools` trên cùng tệp COCO — đúng bộ
đo đã dùng cho v2, nên hai bảng ghép chung được.

Mỗi cấu hình được báo cáo **song song trên tập đầy đủ và tập sạch**, vì rò rỉ ảnh hưởng nặng
hơn hẳn tới đúng chỉ số quan trọng nhất (`AP` nhỏ).

In [ ]:
from pycocotools.coco import COCO as PyCOCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

TEST_META = json.loads((COCO / "instances_test.json").read_text())
TEST_IMAGES = TEST_META["images"]
print(f"tap kiem tra: {len(TEST_IMAGES)} anh | {len(TEST_META['annotations'])} hop"
      f" | sach: {len(CLEAN_TEST_IDS)}")


class Adapter:
    def __init__(self, name, path, device, fmt, precision, role):
        self.name, self.path = name, Path(path)
        self.device, self.fmt = device, fmt
        self.precision, self.role = precision, role
        self.m = YOLO(str(path), task="detect")

    def predict(self, pil):
        r = self.m.predict(pil, conf=CONF_EVAL, iou=0.7, max_det=300,
                           imgsz=IMGSZ, device=self.device, verbose=False)[0]
        b = r.boxes
        if b is None or len(b) == 0:
            return np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int)
        return (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(),
                b.cls.cpu().numpy().astype(int) + 1)

    def n_params(self):
        try:
            return sum(p.numel() for p in self.m.model.parameters())
        except Exception:
            return float("nan")

    def size_mb(self):
        if self.path.is_dir():
            return sum(f.stat().st_size for f in self.path.rglob("*") if f.is_file()) / 1024**2
        return self.path.stat().st_size / 1024**2


def coco_eval(gt_dict, dets, img_ids=None):
    gt = {k: v for k, v in gt_dict.items()}
    gt["images"] = [{k: v for k, v in im.items() if k != "abs_path"} for im in gt_dict["images"]]
    if not dets:
        return None, None
    with contextlib.redirect_stdout(io.StringIO()):
        c = PyCOCO(); c.dataset = gt; c.createIndex()
        e = COCOeval(c, c.loadRes(list(dets)), "bbox")
        if img_ids is not None:
            e.params.imgIds = sorted(img_ids)
        e.evaluate(); e.accumulate(); e.summarize()
    s = e.stats
    return {"map50_95": float(s[0]), "map50": float(s[1]), "map75": float(s[2]),
            "map_small": float(s[3]), "map_medium": float(s[4]), "map_large": float(s[5]),
            "recall_100": float(s[8])}, e


def measure_speed(ad, warmup=N_SPEED_WARMUP, runs=N_SPEED_RUNS):
    pool = [Image.open(im["abs_path"]).convert("RGB")
            for im in TEST_IMAGES[:max(warmup, runs)]]
    for i in range(warmup):
        ad.predict(pool[i % len(pool)])
    if ad.device != "cpu":
        torch.cuda.synchronize()
    ts = []
    for i in range(runs):
        t0 = time.perf_counter()
        ad.predict(pool[i % len(pool)])
        if ad.device != "cpu":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    for p in pool:
        p.close()
    ts = np.array(ts)
    med = float(np.median(ts))
    return {"latency_median_ms": round(med, 2),
            "latency_p95_ms": round(float(np.percentile(ts, 95)), 2),
            "fps": round(1000 / med, 1)}


def run_inference(ad, images=TEST_IMAGES):
    dets, t0 = [], time.time()
    for i, im in enumerate(images, 1):
        with Image.open(im["abs_path"]) as pil:
            b, s, l = ad.predict(pil.convert("RGB"))
        for (x1, y1, x2, y2), sc, c in zip(b, s, l):
            dets.append({"image_id": im["id"], "category_id": int(c),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                         "score": float(sc)})
        if i % 200 == 0 or i == len(images):
            print(f"   [{ad.name}] {i}/{len(images)} ({time.time() - t0:.0f}s)")
    return dets


def evaluate(ad, with_speed=True):
    print(f"\n{'=' * 64}\n  {ad.name}\n{'=' * 64}")
    dets = run_inference(ad)
    overall, _ = coco_eval(TEST_META, dets)
    if overall is None:
        print("  !! khong co du doan nao")
        return None

    clean = {}
    if LEAKED_TEST_IDS and len(CLEAN_TEST_IDS) >= 10:
        co, _ = coco_eval(TEST_META, dets, img_ids=CLEAN_TEST_IDS)
        if co:
            clean = {f"clean_{k}": v for k, v in co.items()}

    spd = measure_speed(ad) if with_speed else {}
    npar = ad.n_params()
    m = {"model": ad.name, "role": ad.role, "format": ad.fmt, "precision": ad.precision,
         "device": str(ad.device), **overall, **clean, **spd,
         "n_test_all": len(TEST_IMAGES), "n_test_clean": len(CLEAN_TEST_IDS),
         "params_m": round(npar / 1e6, 2) if npar == npar else None,
         "size_mb": round(ad.size_mb(), 2), "imgsz": IMGSZ}
    (RESULTS / "metrics" / f"{ad.name.replace(' ', '_').replace('/', '_').lower()}.json"
     ).write_text(json.dumps(m, indent=2))

    print(f"  mAP@0.5        : {m['map50']:.4f}"
          + (f"   | sach: {m['clean_map50']:.4f}" if clean else ""))
    print(f"  mAP@0.5:0.95   : {m['map50_95']:.4f}")
    print(f"  AP nho/vua/lon : {m['map_small']:.3f} / {m['map_medium']:.3f} / {m['map_large']:.3f}")
    if clean:
        print(f"  AP nho (sach)  : {m['clean_map_small']:.3f}")
    if spd:
        print(f"  do tre trung vi: {spd['latency_median_ms']} ms (p95 {spd['latency_p95_ms']})")
    print(f"  kich thuoc     : {m['params_m']} M | {m['size_mb']} MB  [{ad.device}]")
    return m


CONFIGS = [
    ("Thay yolo26s",       TEACHER_BEST,  DEVICE, "pytorch", "FP16", "teacher"),
    ("Tro doi chung",      BASELINE_BEST, DEVICE, "pytorch", "FP16", "student_baseline"),
    (f"Tro chung cat dis{DIS_WEIGHT_MAIN}", KD_BEST, DEVICE, "pytorch", "FP16", "student_kd"),
    ("ONNX FP32",          KD_ONNX_FP32,  "cpu",  "onnx",    "FP32", "student_kd_onnx"),
    ("ONNX INT8",          KD_ONNX_INT8,  "cpu",  "onnx",    "INT8", "student_kd_onnx"),
]

all_metrics = []
for nm, pth, dev, fmt, prec, role in CONFIGS:
    try:
        r = evaluate(Adapter(nm, pth, dev, fmt, prec, role))
        if r:
            all_metrics.append(r)
    except Exception as ex:
        print(f"\n!! BO QUA {nm}: {type(ex).__name__}: {ex}")

print(f"\nDa danh gia {len(all_metrics)}/{len(CONFIGS)} cau hinh chinh.")

### 5.1 Đánh giá các nhánh trong phép quét

Chỉ đo độ chính xác, **không đo tốc độ** — mọi nhánh quét đều là `yolo26n` với đúng số tham số
như nhau nên tốc độ không thể khác; đo lại chỉ tốn thời gian.

In [ ]:
sweep_metrics = []
for dw in sorted(sweep_runs):
    if dw == DIS_WEIGHT_MAIN:
        base = next((m for m in all_metrics if m["role"] == "student_kd"), None)
        if base:
            sweep_metrics.append({**base, "dis": dw})
            continue
    try:
        r = evaluate(Adapter(f"KD dis={dw}", sweep_runs[dw], DEVICE, "pytorch", "FP16",
                             "student_kd_sweep"), with_speed=False)
        if r:
            sweep_metrics.append({**r, "dis": dw})
    except Exception as ex:
        print(f"!! BO QUA dis={dw}: {ex}")

print(f"\nDa danh gia {len(sweep_metrics)} nhanh trong phep quet.")

---
## §6 · Kết quả

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)

rows = []
for m in all_metrics:
    rows.append({
        "Cấu hình": m["model"], "Vai trò": m["role"],
        "Tệp (MB)": m["size_mb"], "Tham số (M)": m["params_m"],
        "mAP@0,5": round(m["map50"], 4), "mAP@.5:.95": round(m["map50_95"], 4),
        "AP nhỏ": round(m["map_small"], 3), "AP lớn": round(m["map_large"], 3),
        "AP nhỏ (sạch)": round(m.get("clean_map_small", float("nan")), 3),
        "Trung vị (ms)": m.get("latency_median_ms"), "FPS": m.get("fps"),
    })
df = pd.DataFrame(rows)
df.to_csv(RESULTS / "tables" / "v3_comparison.csv", index=False)
print(df.to_string(index=False))
df

### 6.1 Đối chiếu v2 ↔ v3 — câu hỏi trung tâm của cả notebook

Nếu **chỉ** batch của nhánh chưng cất thay đổi mà kết quả âm biến mất, thì kết luận đúng của
báo cáo phải là *"chênh lệch ở v2 do yếu tố gây nhiễu"*. Nếu kết quả âm vẫn còn, thì kết luận
*"chưng cất không mang lại lợi ích trong cấu hình này"* được củng cố mạnh hơn hẳn — vì phản
biện hiển nhiên nhất đã bị loại bỏ.

Cả hai khả năng đều là kết quả **có giá trị báo cáo**.

In [ ]:
def pick(role):
    return next((m for m in all_metrics if m["role"] == role), None)


T, B, K = pick("teacher"), pick("student_baseline"), pick("student_kd")
KF = next((m for m in all_metrics if m["precision"] == "FP32"), None)
KI = next((m for m in all_metrics if m["precision"] == "INT8"), None)

_map_role = {"teacher": T, "student_baseline": B, "student_kd": K,
             "onnx_fp32": KF, "onnx_int8": KI}

cmp_rows = []
for key, new in _map_role.items():
    old = V2_RESULTS[key]
    if not new:
        continue
    cmp_rows.append({
        "Cấu hình": key,
        "mAP@0,5 v2": old["map50"], "mAP@0,5 v3": round(new["map50"], 4),
        "Δ mAP": round(new["map50"] - old["map50"], 4),
        "AP nhỏ v2": old["ap_small"], "AP nhỏ v3": round(new["map_small"], 4),
        "Δ AP nhỏ": round(new["map_small"] - old["ap_small"], 4),
    })
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(RESULTS / "tables" / "v2_vs_v3.csv", index=False)
print(cmp_df.to_string(index=False))

print("\n" + "=" * 68)
print("  CAU HOI TRUNG TAM: chung cat con am khong khi batch da bang nhau?")
print("=" * 68)
if B and K:
    d50 = K["map50"] - B["map50"]
    d5095 = K["map50_95"] - B["map50_95"]
    dsm = K["map_small"] - B["map_small"]
    print(f"  v2 (batch 8 vs 24) : Δ mAP@0,5 = -0.0334 | Δ AP nho = -0.0620")
    print(f"  v3 (batch bang nhau): Δ mAP@0,5 = {d50:+.4f} | Δ AP nho = {dsm:+.4f}")
    print()
    if d5095 > 0:
        print("  => CHUNG CAT DA CO LOI khi loai bo yeu to gay nhieu.")
        print("     Ket luan v2 ('chung cat khong co loi') phai duoc sua lai:")
        print("     chenh lech cu chu yeu den tu batch size, khong phai tu chung cat.")
    elif d5095 >= THRESH["kd_not_worse_than_baseline"]:
        print("  => CHUNG CAT HOA VON (trong nguong chap nhan).")
        print("     Loi ich khong ro rang, nhung cung khong con ket qua am.")
    else:
        print("  => CHUNG CAT VAN AM du batch da bang nhau.")
        print("     Ket luan v2 duoc CUNG CO: phan bien 'tai batch size' da bi loai bo,")
        print("     nen nguyen nhan con lai la khoang cach nang luc thay-tro qua hep.")
print("=" * 68)

### 6.2 Kết quả phép quét hệ số chưng cất

In [ ]:
if sweep_metrics:
    sw = pd.DataFrame([{
        "dis": m["dis"], "mAP@0,5": round(m["map50"], 4),
        "mAP@.5:.95": round(m["map50_95"], 4),
        "AP nhỏ": round(m["map_small"], 3), "AP lớn": round(m["map_large"], 3),
        "Δ so đối chứng (mAP@.5:.95)": round(m["map50_95"] - B["map50_95"], 4) if B else None,
    } for m in sorted(sweep_metrics, key=lambda x: x["dis"])])
    sw.to_csv(RESULTS / "tables" / "dis_sweep.csv", index=False)
    print(sw.to_string(index=False))

    best = max(sweep_metrics, key=lambda m: m["map50_95"])
    print(f"\ndis tot nhat theo mAP@.5:.95 : {best['dis']}  ({best['map50_95']:.4f})")
    if B:
        if best["map50_95"] > B["map50_95"]:
            print(f"-> VUOT doi chung ({B['map50_95']:.4f}). Gia thuyet 'dis=6,0 qua lon' DUNG:")
            print("   chung cat co loi, nhung chi o he so nho hon mac dinh.")
        else:
            print(f"-> Van khong vuot doi chung ({B['map50_95']:.4f}) o BAT KY he so nao da thu.")
            print("   Gia thuyet 'dis qua lon' bi BAC BO — nguyen nhan nam o cho khac.")

    fig, ax = plt.subplots(figsize=(9, 5))
    xs = [m["dis"] for m in sorted(sweep_metrics, key=lambda x: x["dis"])]
    for key, lab, col in [("map50_95", "mAP@.5:.95", "#1F3864"),
                          ("map_small", "AP nhỏ", "#D9821B")]:
        ys = [m[key] for m in sorted(sweep_metrics, key=lambda x: x["dis"])]
        ax.plot(xs, ys, "o-", label=lab, color=col, linewidth=2)
    if B:
        ax.axhline(B["map50_95"], ls="--", color="#1F3864", alpha=0.5,
                   label="đối chứng mAP@.5:.95")
        ax.axhline(B["map_small"], ls="--", color="#D9821B", alpha=0.5,
                   label="đối chứng AP nhỏ")
    ax.set_xlabel("hệ số chưng cất (dis)")
    ax.set_ylabel("điểm")
    ax.set_title("Ảnh hưởng của hệ số chưng cất")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS / "plots" / "dis_sweep.png", dpi=150)
    plt.close(fig)
    print(f"\nbieu do -> {RESULTS / 'plots' / 'dis_sweep.png'}")
else:
    print("chua co ket qua quet")

### 6.3 Lượng tử hoá — bây giờ so được trong cùng một môi trường

Đây là con số mà v2 **không** khẳng định được. Cả FP32 lẫn INT8 đều xuất và đo trong phiên này,
nên tỉ số "chi phí dồn vào vật thể nhỏ" lần đầu tiên có cơ sở tái lập.

In [ ]:
if KF and KI:
    d_map = KF["map50"] - KI["map50"]        # duong = INT8 kem hon FP32
    d_small = KF["map_small"] - KI["map_small"]
    size_red = 1 - KI["size_mb"] / KF["size_mb"]

    # Ti so "chi phi don vao vat the nho" chi co nghia khi CA HAI deu that su sut.
    # Neu luong tu hoa tinh co lam mAP tong TANG (d_map <= 0), phep chia cho ra so
    # am hoac vo cuc va dien giai theo no la sai.
    _valid_ratio = d_map > 1e-4 and d_small > 1e-4
    ratio = d_small / d_map if _valid_ratio else float("nan")

    q = pd.DataFrame([{
        "Phép so sánh": "INT8 so với FP32",
        "Δ mAP@0,5": round(-d_map, 4),
        "Δ AP nhỏ": round(-d_small, 4),
        "Tỉ số (nhỏ / tổng)": round(ratio, 2),
        "Giảm dung lượng": f"{size_red:.1%}",
        "Δ độ trễ (ms)": round(KI.get("latency_median_ms", 0) - KF.get("latency_median_ms", 0), 2),
    }])
    q.to_csv(RESULTS / "tables" / "quantization.csv", index=False)
    print(q.to_string(index=False))

    print(f"\n  v2 bao cao ti so : 6,5 lan (dua tren MOT lan do, FP32 khong tai lap duoc)")
    if _valid_ratio:
        print(f"  v3 do lai        : {ratio:.2f} lan (ca hai ban do trong CUNG phien)")
        if ratio > 1.5:
            print("\n  => Luan diem trung tam DUNG VUNG: chi phi luong tu hoa van don")
            print("     vao vat the nho nang hon han muc ma mAP tong the hien.")
        else:
            print("\n  => Ti so thap hon han v2 — can xem lai ket luan ve luong tu hoa.")
    else:
        print("  v3 do lai        : KHONG tinh duoc ti so")
        print(f"     (Δ mAP tong = {-d_map:+.4f}, Δ AP nho = {-d_small:+.4f})")
        print("\n  => Lan nay luong tu hoa KHONG lam sut ca hai chi so, nen phep chia")
        print("     'gap bao nhieu lan' khong con y nghia. Bao cao thang hai muc sut")
        print("     thay vi ep ra mot ti so.")
else:
    print("thieu ban FP32 hoac INT8 -> khong so sanh duoc")

### 6.4 Kiểm tra các ngưỡng chấp nhận

Ngưỡng đã khai báo ở ô cấu hình §0, **trước** khi thấy bất kỳ số liệu nào. Không đạt ngưỡng
không phải thất bại — điều không chấp nhận được là sửa ngưỡng sau khi đã thấy kết quả.

In [ ]:
checks = []
if B and K:
    d = K["map50_95"] - B["map50_95"]
    checks.append(("KD không kém đối chứng quá 0,005 điểm",
                   f"Δ = {d:+.4f}", d >= THRESH["kd_not_worse_than_baseline"]))
    checks.append(("KD tốt hơn đối chứng (kỳ vọng ban đầu)",
                   f"Δ = {d:+.4f}", d > 0))
if KF and KI:
    d = KF["map50_95"] - KI["map50_95"]
    checks.append(("INT8 giảm không quá 0,015 điểm so với FP32",
                   f"giảm {d:.4f}", d <= THRESH["int8_max_drop"]))
    r = 1 - KI["size_mb"] / KF["size_mb"]
    checks.append((f"INT8 nhỏ hơn FP32 ít nhất {THRESH['int8_min_size_reduction']:.0%}",
                   f"giảm {r:.1%}", r >= THRESH["int8_min_size_reduction"]))
if K and T:
    checks.append(("Trò chưng cất nhẹ hơn thầy",
                   f"{K['size_mb']} vs {T['size_mb']} MB", K["size_mb"] < T["size_mb"]))

print(f"{'Ngưỡng':<52} {'Giá trị đo':<20} {'Kết quả'}")
print("-" * 88)
for name, val, ok in checks:
    print(f"{name:<52} {val:<20} {'ĐẠT' if ok else 'KHÔNG ĐẠT'}")
n_pass = sum(1 for _, _, ok in checks if ok)
print("-" * 88)
print(f"{n_pass}/{len(checks)} ngưỡng đạt")

(RESULTS / "metrics" / "thresholds.json").write_text(json.dumps(
    [{"name": n, "value": v, "passed": bool(o)} for n, v, o in checks], indent=2,
    ensure_ascii=False))

---
## §7 · Đóng gói kết quả

Colab **xoá sạch máy ảo khi ngắt phiên**. Ô này nén toàn bộ trọng số, số liệu và biểu đồ rồi
(nếu đã gắn Drive) chép sang Drive. Chạy ô này trước khi đóng tab.

In [ ]:
STAMP = time.strftime("%Y%m%d_%H%M")
BUNDLE = WORK / f"v3_results_{STAMP}"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
(BUNDLE / "weights").mkdir(parents=True)

for name, p in [("teacher_yolo26s.pt", TEACHER_BEST),
                ("student_baseline_yolo26n.pt", BASELINE_BEST),
                ("student_kd_yolo26n.pt", KD_BEST),
                ("student_kd_fp32.onnx", KD_ONNX_FP32),
                ("student_kd_int8.onnx", KD_ONNX_INT8)]:
    if Path(p).exists():
        shutil.copy2(p, BUNDLE / "weights" / name)

for dw, p in sweep_runs.items():
    if dw != DIS_WEIGHT_MAIN and Path(p).exists():
        shutil.copy2(p, BUNDLE / "weights" / f"student_kd_dis{str(dw).replace('.', 'p')}.pt")

shutil.copytree(RESULTS, BUNDLE / "results", dirs_exist_ok=True)
shutil.copy2(OUT / "environment.json", BUNDLE / "environment.json")

for run in ("teacher", "baseline", "kd"):
    src_csv = RUNS / run / "results.csv"
    if src_csv.exists():
        shutil.copy2(src_csv, BUNDLE / "results" / f"trainlog_{run}.csv")

ZIP = shutil.make_archive(str(BUNDLE), "zip", BUNDLE)
print(f"da nen: {ZIP}  ({Path(ZIP).stat().st_size / 1024**2:.1f} MB)")

drive_dir = Path("/content/drive/MyDrive")
if drive_dir.exists():
    dst = drive_dir / Path(ZIP).name
    shutil.copy2(ZIP, dst)
    print(f"da chep sang Drive: {dst}")
else:
    print("Drive chua duoc gan — tai thu cong tu bang File ben trai.")
    if PLATFORM == "colab":
        try:
            from google.colab import files
            files.download(ZIP)
        except Exception as ex:
            print("tu dong tai that bai:", ex)

---
## Việc cần làm sau khi chạy xong

1. **Đọc ô §6.1 trước tiên.** Nó trả lời thẳng câu hỏi "kết quả âm của chưng cất có phải do
   batch size không". Dù ra chiều nào cũng phải sửa lại Mục 4.2.2 và 5.3 của báo cáo cho khớp.
2. **Đọc ô §6.2.** Nếu có một hệ số `dis` nào vượt được đối chứng, đó là phát hiện mới đáng kể
   và Ưu tiên 3 trong Mục 5.4 coi như đã hoàn thành.
3. **Đọc ô §6.3.** Tỉ số "chi phí dồn vào vật thể nhỏ" giờ đã tái lập được — gạch bỏ được dòng
   "phép so INT8 với FP32 chưa tái lập độc lập" khỏi phần hạn chế.
4. **Đưa đính chính BatchNorm ở đầu notebook vào báo cáo**, bất kể kết quả ra sao. Nó làm phần
   hạn chế chính xác hơn hẳn so với cách diễn đạt hiện tại.

Những hạn chế v3 **vẫn chưa** giải quyết, cần giữ nguyên trong báo cáo: chỉ khảo sát một cặp
thầy–trò, lượng tử hoá chỉ đo trên một môi trường thực thi, tập lớp giới hạn ở 15 lớp, và ba
video kiểm chứng vẫn chưa có nhãn thật.
